# MTL — Colab/Kaggle GPU training

Run cells top to bottom. This trains the shared ResNet50+FPN backbone
jointly on detection (RetinaNet) + semantic segmentation (FCN) +
multi-label classification, on a COCO subset.


## 1. Install dependencies
Colab usually ships a CUDA-matched torch/torchvision already — only reinstall if missing.

In [1]:
import torch
print(torch.__version__, torch.cuda.is_available())

# Uncomment only if the above shows no CUDA:
# !pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121

# timm: DINO ViT-B/16 omurgası için (configs/train_colab_dino.yaml). torch/torchvision'a dokunmaz.
!pip install pycocotools PyYAML tqdm scikit-learn requests timm

2.11.0+cu128 True


## 2. Get the repo
Either clone from git, or upload a zip of this project via the Colab file browser and unzip it.

In [2]:
# dino-backbone dalını klonla (DINO kodu + bu notebook bu dalda).
# main'e merge ettikten sonra `-b dino-backbone`'u kaldırabilirsin.
!git clone -b dino-backbone https://github.com/itu-itis23-ucgun22/mtl.git
%cd mtl
!pip install -e .

Cloning into 'mtl'...
remote: Enumerating objects: 430, done.
remote: Counting objects: 100% (430/430), done.
remote: Compressing objects: 100% (244/244), done.
remote: Total 430 (delta 250), reused 310 (delta 141), pack-reused 0 (from 0)
Receiving objects: 100% (430/430), 802.78 KiB | 9.02 MiB/s, done.
Resolving deltas: 100% (250/250), done.
/content/mtl
Obtaining file:///content/mtl
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for mtl (pyproject.toml) ... done
  Created wheel for mtl: filename=mtl-0.1.0-0.editable-py3-none-any.whl size=1181 sha256=d111b6072179356b9303472962e93d4bcee5d9cf770424a6fa3a1475e25d529d
  Stored in directory: /tmp/pip-ephem-wheel-cache-qvu_dlkk/wheels/b8/3b/63/cd9bbc919e23d84900281b7352d08ba8b6890722307e161497
Successfully built mtl


In [ ]:
!git -C /content/mtl pull


shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
^C


## 2.5. Google Drive'ı bağla (veri + checkpoint'i kalıcı yap)
COCO subset'i **ve** `checkpoints/`'i Drive'da tutar (ikisi de symlink olur). Veri: ilk oturumda section 3 indirir, sonrakilerde Drive'dan gelir — baştan indirmez. Checkpoint: `train.py`'nin yazdığı `.pt` dosyaları doğrudan Drive'a düşer, runtime kopunca kaybolmaz; sonraki oturumda `--resume` ile devam edilebilir.

In [3]:
# --- Google Drive: COCO subset'i kalıcı yap (her oturumda baştan indirmemek için) ---
from google.colab import drive
drive.mount('/content/drive')

import os, glob, shutil

# ResNet denemesinde veriyi zaten Drive'a koymuştun -> otomatik bul, yeniden indirme yok.
# (Yolunu biliyorsan aramayı atlamak için DRIVE_ROOT'u doğrudan o klasöre ayarla.)
DRIVE_ROOT = '/content/drive/MyDrive/mtl/data/coco_subset/annotations'

default = f'{DRIVE_ROOT}/coco_subset/annotations/instances_train_subset.json'
if os.path.exists(default):
    target = f'{DRIVE_ROOT}/coco_subset'
    print('Drive\'da mevcut veri (varsayılan konum):', target)
else:
    # Tüm MyDrive'ı tarayıp subset annotasyonunu bul (bir kerelik, biraz sürebilir).
    hits = glob.glob('/content/drive/MyDrive/**/instances_train_subset.json', recursive=True)
    if hits:
        target = os.path.dirname(os.path.dirname(hits[0]))  # .../coco_subset
        print('Drive\'da mevcut veri bulundu:', target)
    else:
        target = f'{DRIVE_ROOT}/coco_subset'
        os.makedirs(target, exist_ok=True)
        print('Drive\'da veri yok; ilk indirme buraya yazacak:', target)

# data/coco_subset -> Drive'daki klasöre symlink
os.makedirs('data', exist_ok=True)
link = 'data/coco_subset'
if os.path.islink(link):
    os.remove(link)
elif os.path.isdir(link):
    shutil.rmtree(link)
os.symlink(target, link)
print('coco_subset ->', os.path.realpath(link))

# --- checkpoint'ler de Drive'da kalsın ki oturum kesilince kaybolmasın ---
# checkpoints/ -> Drive'daki kalıcı klasöre symlink. Böylece train.py'nin yazdığı
# _epoch*.pt / _step*.pt dosyaları doğrudan Drive'a düşer; runtime kopunca kaybolmaz
# ve sonraki oturumda --resume ile kaldığın yerden devam edebilirsin.
CKPT_DRIVE = '/content/drive/MyDrive/mtl_data/checkpoints'
os.makedirs(CKPT_DRIVE, exist_ok=True)
ckpt_link = 'checkpoints'
if os.path.islink(ckpt_link):
    os.remove(ckpt_link)
elif os.path.isdir(ckpt_link):
    # Bu oturumda yerelde birikmiş checkpoint varsa önce Drive'a taşı, sonra symlink'e çevir.
    for f in glob.glob(f'{ckpt_link}/*'):
        dst = os.path.join(CKPT_DRIVE, os.path.basename(f))
        if not os.path.exists(dst):
            shutil.move(f, dst)
    shutil.rmtree(ckpt_link)
os.symlink(CKPT_DRIVE, ckpt_link)
print('checkpoints ->', os.path.realpath(ckpt_link))
print('  mevcut:', sorted(os.listdir(CKPT_DRIVE)) or '(bos)')

Mounted at /content/drive
Drive'da mevcut veri bulundu: /content/drive/MyDrive/mtl/data/coco_subset
coco_subset -> /content/drive/MyDrive/mtl/data/coco_subset
checkpoints -> /content/drive/.shortcut-targets-by-id/1QcCNm9lZ7ZgvWmWW23akax-rBArOZILh/mtl_data/checkpoints
  mevcut: ['colab_beit_cached_epoch0.pt', 'colab_beit_cached_epoch1.pt', 'colab_mae_lora_epoch0.pt', 'colab_mae_lora_step1000.pt', 'colab_mae_lora_step1500.pt', 'colab_mae_lora_step2000.pt', 'colab_mae_lora_step2500.pt', 'colab_mae_lora_step3000.pt', 'colab_mae_lora_step3500.pt', 'colab_mae_lora_step4000.pt', 'colab_mae_lora_step4500.pt', 'colab_mae_lora_step500.pt', 'colab_mae_lora_step5000.pt', 'colab_mae_lora_step5500.pt', 'colab_mae_lora_step6000.pt', 'colab_mae_lora_step6500.pt', 'colab_mae_lora_step7000.pt', 'colab_mae_lora_step7500.pt', 'colab_mae_lora_step8000.pt', 'colab_mae_lora_step8500.pt', 'colab_mae_lora_step9000.pt', 'colab_mae_lora_step9500.pt']


## 3. COCO subset'i hazırla
İlk oturumda full COCO annotations'ı indirip subset'i (22.5k train / 2k val) oluşturur ve görüntüleri Drive'a indirir. Section 2.5 sayesinde **veri zaten Drive'daysa bu hücre indirmeyi atlar.**

In [ ]:
%%bash
set -e
# Veri Drive'da zaten varsa (section 2.5 symlink'i) tüm indirmeyi atla.
if [ -f data/coco_subset/annotations/instances_train_subset.json ] \
   && [ -f data/coco_subset/annotations/instances_val_subset.json ] \
   && [ -n "$(ls -A data/coco_subset/images/train 2>/dev/null)" ] \
   && [ -n "$(ls -A data/coco_subset/images/val 2>/dev/null)" ]; then
  echo "Veri Drive'da mevcut -> indirme atlanıyor."
  exit 0
fi

# Full COCO annotations (subset JSON'u üretmek için gerekli) - subset zaten varsa buraya girilmez.
wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip
unzip -q -o annotations_trainval2017.zip

python scripts/prepare_coco_subset.py \
    --ann-file annotations/instances_train2017.json \
    --out data/coco_subset/annotations/instances_train_subset.json \
    --n-images 22500
python scripts/prepare_coco_subset.py \
    --ann-file annotations/instances_val2017.json \
    --out data/coco_subset/annotations/instances_val_subset.json \
    --n-images 2000

# Sadece subset'in görüntülerini indir (script mevcut dosyaları atlar -> Drive'da varsa hızlı geçer).
python scripts/download_subset_images.py \
    --ann-file data/coco_subset/annotations/instances_train_subset.json \
    --out-dir data/coco_subset/images/train
python scripts/download_subset_images.py \
    --ann-file data/coco_subset/annotations/instances_val_subset.json \
    --out-dir data/coco_subset/images/val

## 3.5. Veriyi yerel diske hazırla — HIZLI zip yolu (eğitim I/O'sunu maksimize et)
Google Drive mount'u (FUSE) binlerce **küçük** görüntüyü rastgele okumakta çok yavaştır ve Google bu deseni **throttle'lar** (20 dk'lık iş 2 saate çıkabilir) → GPU veri beklerken boşta kalır, bütçe yanar. Çözüm: görüntüleri Drive'da **tek `.zip`** olarak tut (bkz. section 0/README) → bu hücre onu **tek büyük dosya** olarak indirir (hızlı sıralı okuma, throttle yok), Colab'ın yerel SSD'sine (`/content/coco_local`) açar ve `data/coco_subset` symlink'ini oraya yöneltir. Annotations (küçük) Drive'dan gelir. Config'ler değişmeden tüm eğitim/eval yerelden okur; Drive orijinali kalıcı depo olarak durur. **Zip yoksa** eski yavaş `copytree` yöntemine otomatik düşer (o zaman bir zip yükleyip hızlandır). Section 3'ten (veri Drive'da hazır) sonra çalıştır.

In [4]:
# --- Veriyi yerel diske (/content) hazırla: HIZLI zip yolu (maks. I/O verimi) ---
# Drive (FUSE) binlerce KÜÇÜK dosyayı rastgele okumakta yavaştır ve throttle olur. Bu yüzden
# görüntüleri Drive'da TEK zip olarak tutuyoruz: tek büyük dosya = hızlı sıralı okuma.
# /content her oturumda silindiği için bu hücre her oturumda bir kez çalışır.
import os, glob, time, shutil

DRIVE_DIR = '/content/drive/MyDrive/mtl/data'
DRIVE_ANN = f'{DRIVE_DIR}/coco_subset/annotations'   # küçük; Drive'da
LOCAL, UNZIP = '/content/coco_local', '/content/coco_images_unzip'

def _relink(root):
    link = 'data/coco_subset'; os.makedirs('data', exist_ok=True)
    if os.path.islink(link): os.remove(link)
    elif os.path.isdir(link): shutil.rmtree(link)
    os.symlink(root, link)
    print('coco_subset ->', os.path.realpath(link))

def _find_split(split):
    dirs = {os.path.dirname(p) for p in glob.glob(f'{UNZIP}/**/*.jpg', recursive=True)
            if f'/{split}/' in p.replace(os.sep, '/') or os.path.basename(os.path.dirname(p)) == split}
    if not dirs:
        raise SystemExit(f"'{split}' görüntü klasörü zip'te bulunamadı — zip yapısını kontrol et")
    return max(dirs, key=lambda d: len(glob.glob(f'{d}/*.jpg')))

if os.path.exists(f'{LOCAL}/annotations/instances_train_subset.json') and os.path.isdir(f'{LOCAL}/images/train'):
    print('Yerel kopya zaten hazır:', LOCAL)
    _relink(LOCAL)
else:
    zips = glob.glob(f'{DRIVE_DIR}/**/*.zip', recursive=True)
    if zips:  # ---- HIZLI YOL: görüntüler tek zip'te ----
        DRIVE_ZIP = max(zips, key=os.path.getsize)   # en büyük zip = görüntüler
        print(f'HIZLI YOL — zip: {DRIVE_ZIP} ({os.path.getsize(DRIVE_ZIP)/1e9:.2f} GB)')
        if not glob.glob(f'{UNZIP}/**/*.jpg', recursive=True):
            t = time.time()
            !cp "{DRIVE_ZIP}" /content/imgs.zip       # tek büyük dosya = hızlı sıralı okuma
            os.makedirs(UNZIP, exist_ok=True)
            !unzip -q -o /content/imgs.zip -d {UNZIP}
            print(f'unzip: {time.time()-t:.0f} sn')
        os.makedirs(f'{LOCAL}/images', exist_ok=True)
        if not os.path.isdir(f'{LOCAL}/annotations'):   # annotations (küçük) Drive'dan
            assert os.path.isdir(DRIVE_ANN), f'annotations Drive\'da yok: {DRIVE_ANN} (section 3 üretir)'
            shutil.copytree(DRIVE_ANN, f'{LOCAL}/annotations')
        for split in ('train', 'val'):                  # images (zip'ten) yerine taşı
            dst = f'{LOCAL}/images/{split}'
            if not os.path.isdir(dst):
                shutil.move(_find_split(split), dst)
        _relink(LOCAL)
        print('train:', len(os.listdir(f'{LOCAL}/images/train')), '| val:', len(os.listdir(f'{LOCAL}/images/val')))
    else:  # ---- YAVAŞ YEDEK: zip yoksa Drive'daki dağınık dosyaları kopyala (eski yöntem) ----
        src = os.path.realpath('data/coco_subset')
        print(f'YAVAŞ YEDEK — Drive copytree (bir zip yükleyip hızlandır): {src} -> {LOCAL}')
        t = time.time()
        shutil.copytree(src, LOCAL, dirs_exist_ok=True)
        print(f'copytree bitti: {time.time()-t:.0f} sn')
        _relink(LOCAL)

HIZLI YOL — zip: /content/drive/MyDrive/mtl/data/coco_subset/zipped.zip (2.14 GB)
unzip: 56 sn
coco_subset -> /content/coco_local
train: 11224 | val: 2000


In [5]:
!python scripts/download_subset_images.py \
    --ann-file data/coco_subset/annotations/instances_train_subset.json \
    --out-dir data/coco_subset/images/train
!python scripts/download_subset_images.py \
    --ann-file data/coco_subset/annotations/instances_val_subset.json \
    --out-dir data/coco_subset/images/val

downloading images: 100% 22500/22500 [05:02<00:00, 74.42it/s]   
Done. 22500/22500 images present in data/coco_subset/images/train.
downloading images: 100% 2000/2000 [00:00<00:00, 80833.02it/s]
Done. 2000/2000 images present in data/coco_subset/images/val.


## 4. Train

### 4.0. (Opsiyonel) GPU kullanımını ölç — I/O darboğazı var mı?
Kısa bir eğitimi arka planda koşup `nvidia-smi` ile GPU kullanımını basar. **Optimizasyon öncesi/sonrası** bir kez koşup kıyasla: `sm` sütunu (GPU hesap %) steady-state'te (~50. adımdan sonra) **%85-100 → darboğaz yok**; sık sık **%0-40'a düşüyorsa → GPU veri bekliyor** (3.5 yerel-kopya hücresini çalıştır). İlk ~45 sn ısınmadır (ağırlık indirme + worker açılışı), ona bakma.

In [ ]:
# --- GPU-util ölçümü: 150 adımlık kısa eğitimi arka planda koş, nvidia-smi ile izle ---
# Amaç metrik değil, GPU meşgul mü diye bakmak. checkpoint_every_steps>150 verip ara
# checkpoint yazmasını da engelliyoruz (bütçe/çöp dosya olmasın).
import subprocess, time

p = subprocess.Popen(
    "python scripts/train.py --config configs/train_colab_gpu.yaml "
    "--overrides train.max_steps=150 train.checkpoint_every_steps=1000",
    shell=True,
)
time.sleep(45)                    # ısınmayı (ağırlık indirme, worker açılışı) geç
!nvidia-smi dmon -s u -c 30       # 30 sn boyunca GPU(sm)/bellek(mem) kullanımı, saniyede bir satır
p.wait()
print("\n[ölçüm bitti] 'sm' sütununa bak: yüksek+sabit = iyi, sık düşüş = veri bekliyor.")

In [ ]:
!python scripts/train.py --config configs/train_colab_gpu.yaml --overrides model.trainable_backbone_layers=0 train.max_steps=2813

## 5. Evaluate + visualize a few predictions

In [ ]:
!python scripts/eval.py --config configs/train_colab_gpu.yaml

In [ ]:
import torch
import matplotlib.pyplot as plt
from torchvision.utils import draw_bounding_boxes

from mtl.config import load_config
from mtl.datasets.coco_multitask import CocoMultiTaskDataset
from mtl.engine.checkpoint import load_checkpoint
from mtl.models.multitask_model import MultiTaskModel
from mtl.utils.device import resolve_device

cfg = load_config("configs/train_colab_gpu.yaml")
device = resolve_device(cfg.train.device)
dataset = CocoMultiTaskDataset(cfg.data.val_ann_file, cfg.data.val_img_dir, img_size=cfg.data.img_size, train=False)

model = MultiTaskModel(
    det_num_classes=dataset.num_classes,
    seg_num_classes=dataset.num_classes + 1,
    cls_num_labels=dataset.num_classes,
).to(device)
load_checkpoint(model, optimizer=None, path="checkpoints/colab_gpu_epoch15.pt", map_location=str(device))
model.eval()

image, target = dataset[0]
with torch.no_grad():
    out = model(image.unsqueeze(0).to(device))

det = out["detections"][0]
keep = det["scores"] > 0.5
img_uint8 = ((image * 0.5 + 0.5) * 255).clamp(0, 255).byte()
drawn = draw_bounding_boxes(img_uint8, det["boxes"][keep].cpu())
plt.imshow(drawn.permute(1, 2, 0))
plt.title(f"top labels: {out['cls_pred'][0].topk(5).indices.tolist()}")
plt.show()

## 6. DINO omurgası (ikinci deney) + ResNet ile karşılaştırma
Aynı pipeline, backbone'da ResNet50+FPN yerine DINOv1 ViT-B/16 + Simple Feature Pyramid. `runs/results.csv` iki omurganın metriklerini biriktirir; `compare_results.py` markdown tablo basar.

In [ ]:
# --- DINO omurgası: tam koşu (çoklu oturum + resume'a dayanıklı) ---
# ResNet yolu aynen train_colab_gpu.yaml ile kalır; DINO sadece backbone'u değiştirir
# (configs/train_colab_dino.yaml). 12 GB VRAM için batch_size=4; sığmazsa aşağıya
# --overrides train.batch_size=2 ekle.
#
# Tam koşu (epochs=16). checkpoint_every_steps=500 ile her 500 adımda checkpoint Drive'a
# düşer. Oturum koparsa BİR SONRAKİ oturumda (aynı ya da paylaşımlı Drive'lı başka hesap)
# en son step-checkpoint'inden devam et -> artık --resume global ADIMdan devam eder,
# sıfırdan değil (scripts/train.py + engine/checkpoint.py). Tamamlanan epoch'lar atlanır.

# 1) İlk oturum - baştan başlat:
!python scripts/train.py --config configs/train_colab_dino.yaml

# 1b) Sonraki oturum(lar) - kaldığın step-checkpoint'inden devam et (en son .pt'yi yaz):
#     (checkpoints/ Drive'a symlink, o yüzden kopan oturumun checkpoint'i burada durur)
# !ls -t checkpoints/colab_dino_step*.pt | head -1     # en son checkpoint'i gör
# !python scripts/train.py --config configs/train_colab_dino.yaml --resume checkpoints/colab_dino_step3000.pt

# 2) Koşu bitince değerlendir (tam 16 epoch -> son checkpoint colab_dino_epoch15.pt;
#    erken durdurduysan en yüksek epoch/step'li .pt'yi yaz). results.csv'ye satır ekler:
!python scripts/eval.py --config configs/train_colab_dino.yaml --checkpoint checkpoints/colab_dino_epoch15.pt

# 3) Adil kıyas: ResNet'i de AYNI epoch'a kadar koşup eval et (elindeki .pt yolunu yaz):
# !python scripts/eval.py --config configs/train_colab_gpu.yaml --checkpoint checkpoints/colab_gpu_epoch15.pt

# 4) ResNet vs DINO karşılaştırma tablosu:
!python scripts/compare_results.py

## 7. DINOv2 omurgası (üçüncü deney) + karşılaştırma
Aynı pipeline, backbone'da DINOv2 ViT-B/14 (+4 register) + Simple Feature Pyramid (`configs/train_colab_dinov2.yaml`, `src/mtl/models/dinov2_backbone.py`). DINOv1 baseline'ı bozmamak için DINOv2 **ayrı dosyada**; head'ler/pipeline değişmez. `runs/results.csv` üç omurganın (ResNet / DINOv1 / DINOv2) metriklerini biriktirir.

**Not:** DINOv2 patch14 olduğu için config'te `img_size: 518` (14'e bölünebilir, modelin native çözünürlüğü). İlk çalıştırmada DINOv2 ağırlıkları HuggingFace'ten iner (birkaç sn). timm zaten section 1'de kurulu.

In [ ]:
# --- DINOv2 omurgası: tam koşu (çoklu oturum + resume'a dayanıklı) ---
# DINOv1 hücresiyle (section 6) birebir aynı akış, tek fark config: train_colab_dinov2.yaml
# (backbone dinov2_vitb14_reg, img_size=518). 12 GB VRAM için batch_size=4; sığmazsa
# --overrides train.batch_size=2 ekle. Hafif/hızlı istersen model.backbone_name=dinov2_vits14.

# 1) İlk oturum - baştan başlat:
!python scripts/train.py --config configs/train_colab_dinov2.yaml

# 1b) Sonraki oturum(lar) - en son step-checkpoint'inden devam et (checkpoints/ Drive'a symlink):
# !ls -t checkpoints/colab_dinov2_step*.pt | head -1     # en son checkpoint'i gör
# !python scripts/train.py --config configs/train_colab_dinov2.yaml --resume checkpoints/colab_dinov2_step3000.pt

# 2) Koşu bitince değerlendir (tam 16 epoch -> colab_dinov2_epoch15.pt). results.csv'ye satır ekler:
!python scripts/eval.py --config configs/train_colab_dinov2.yaml --checkpoint checkpoints/colab_dinov2_epoch15.pt

# 3) Üç omurga karşılaştırma tablosu (ResNet / DINOv1 / DINOv2 - hepsi results.csv'de):
!python scripts/compare_results.py

## 8. CLIP omurgası (dördüncü deney) — feature-caching ile + karşılaştırma
Aynı pipeline, backbone'da **CLIP ViT-B/16 (OpenAI)** + Simple Feature Pyramid (`configs/train_colab_clip.yaml`, `src/mtl/models/clip_backbone.py`). CLIP **patch16** olduğu için DINOv1 ile **aynı grid** (512px → 32×32) → "aynı ViT mimarisi, farklı pretraining (**dil-contrastive** vs SSL)" en temiz kıyas noktası (patch confound'u yok). DINOv1 baseline'ı bozmamak için **ayrı dosya**; head'ler/pipeline değişmez.

Donuk (layers=0) olduğu için ROADMAP varsayılanı **feature-caching**: donuk CLIP trunk'ı bir kez diske yaz, sonra sadece neck+head'i cache'den hızlı eğit. **Normalizasyon** backbone içinde ImageNet→CLIP'e çevrilir (CLIP kendi mean/std'siyle eğitildi; `transforms.py`'ye dokunulmaz, diğer backbone'lar etkilenmez). `runs/results.csv` dört omurgayı (ResNet / DINOv1 / DINOv2 / CLIP) biriktirir.

**Not:** İlk çalıştırmada CLIP ağırlıkları HuggingFace/timm'den iner (birkaç sn). timm section 1'de kurulu.

In [ ]:
# --- CLIP omurgası: feature-caching akışı (donuk backbone -> ROADMAP varsayılanı) ---
# DINOv1/v2 hücreleriyle aynı pipeline, tek fark config: train_colab_clip.yaml
# (backbone clip_vitb16, img_size=512). CLIP donuk trunk çıktısı deterministik olduğu için
# bir kez cache'lenir, sonra sadece neck+head cache'den HIZLI eğitilir (pahalı ViT forward'ı
# her adımda tekrar koşmaz). Normalizasyon backbone içinde ImageNet->CLIP'e çevrilir.

# 1) Trunk feature'larını bir kez yaz (train split; ViT-B/16@512 -> ~34 GB, /content yerel SSD):
!python scripts/precompute_features.py --config configs/train_colab_clip.yaml --split train

# 2) Neck+head'i cache'den eğit (pahalı ViT forward YOK; checkpoint -> checkpoints/colab_clip_cached_epoch{N}.pt):
!python scripts/train_cached.py --config configs/train_colab_clip.yaml

# 3) Değerlendir (checkpoint tam model state'i -> normal eval.py; görüntü-tabanlı forward). results.csv'ye satır ekler:
!python scripts/eval.py --config configs/train_colab_clip.yaml --checkpoint checkpoints/colab_clip_cached_epoch15.pt

# 4) Dört omurga karşılaştırma tablosu (ResNet / DINOv1 / DINOv2 / CLIP - hepsi results.csv'de):
!python scripts/compare_results.py

# --- (Alternatif) cache YERİNE doğrudan görüntüden eğitim (section 6/7 ile aynı desen) ---
# Feature-caching'de sorun çıkarsa ya da /content'te cache için yer yoksa bu kanıtlanmış yolu kullan:
# !python scripts/train.py --config configs/train_colab_clip.yaml
# !python scripts/eval.py --config configs/train_colab_clip.yaml --checkpoint checkpoints/colab_clip_epoch15.pt

## 9. SAM omurgası (beşinci deney) — feature-caching ile + karşılaştırma
Aynı pipeline, backbone'da **SAM image encoder** (timm `samvit_base_patch16.sa1b`) + Simple Feature Pyramid (`configs/train_colab_sam.yaml`, `src/mtl/models/sam_backbone.py`). Sweep'in **segmentation-native** paradigma temsilcisi — SAM, milyarlarca maskeyle (SA-1B) eğitildiği için dense/spatial görevlerde güçlü olması beklenir; bu deney onu test eder. Donuk (layers=0) → **feature-caching** (CLIP ile aynı akış).

**⚠️ Sanity:** SAM native çözünürlük **1024** (pos-embed 64×64); burada `img_size: 512` ile kuruyoruz, timm pos-embed'i 32×32'ye interpole ediyor. İlk precompute'ta trunk şekli basılır — **(embed_dim, 32, 32)** civarı beklenir. Hata/anlamsız sonuç olursa `sam_backbone.py`'de `SAM_IMG=1024` + config `img_size: 1024` yap (ağır ama native). `runs/results.csv` beş omurgayı biriktirir.

In [ ]:
# --- SAM omurgası (segmentation-native): feature-caching akışı ---
# Gövde: SAM image encoder (timm samvit_base_patch16.sa1b) + Simple Feature Pyramid.
# Donuk trunk deterministik -> cache. CLIP ile aynı akış, tek fark config.
#
# ⚠️ SANITY: 1. adımda trunk şekli basılır — SAM native 1024, burada img_size=512 (pos-embed
# interp). Şekil (embed_dim, 32, 32) civarı beklenir. Hata/anlamsız sonuç olursa
# sam_backbone.py'de SAM_IMG=1024 + config img_size=1024 yap (ağır ama native, pencere hizası bozulmaz).

# 1) Trunk feature'larını yaz (train split):
!python scripts/precompute_features.py --config configs/train_colab_sam.yaml --split train

# 2) Neck+head'i cache'den eğit (fp32 -> focal-loss NaN'ından kaçın):
!python scripts/train_cached.py --config configs/train_colab_sam.yaml --no-amp --resume checkpoints/colab_sam_cached_epoch14.pt

# 3) Değerlendir + karşılaştırma:
#!python scripts/eval.py --config configs/train_colab_sam.yaml --checkpoint checkpoints/colab_sam_cached_epoch15.pt
#!python scripts/compare_results.py

# --- (Alternatif) cache sorun çıkarırsa doğrudan görüntüden eğitim ---
# !python scripts/train.py --config configs/train_colab_sam.yaml --overrides train.amp=false
# !python scripts/eval.py --config configs/train_colab_sam.yaml --checkpoint checkpoints/colab_sam_epoch15.pt

## 10. I-JEPA omurgası (altıncı deney, opsiyonel) — predictive SSL
Aynı pipeline, backbone'da **I-JEPA ViT-H/14** (`configs/train_colab_ijepa.yaml`, `src/mtl/models/ijepa_backbone.py`). I-JEPA maskeli-latent tahminiyle eğitilir (kontrastif/distillation değil) → farklı bir SSL paradigması. patch14 → `img_size: 518` (37×37 grid, DINOv2 gibi).

**⚠️ İki önemli fark:**
1. **`transformers` gerekir** (timm'de temiz tag yok) — hücre `pip install transformers` yapar; ağırlık HF'ten iner (`facebook/ijepa_vith14_1k`).
2. **Sadece ViT-H (632M) var** → diğerleri ViT-B (~86M). Bu bir **BOYUT CONFOUND'u**: I-JEPA öne çıkarsa "paradigma mı 7× büyük model mi" ayrışmaz — sonucu **bu notla** raporla. Ayrıca ViT-H feature-cache **~80 GB** (`/content`'e sığmayabilir) ve VRAM ağır (`batch_size: 2`). Sığmazsa hücredeki **düz `train.py`** alternatifine geç.

Normalizasyon ImageNet (CLIP'teki gibi yeniden ölçekleme gerekmez).

In [ ]:
!git -C /content/mtl pull


Already up to date.


In [ ]:
!python scripts/eval.py \
    --config configs/train_colab_ijepa.yaml \
    --checkpoint checkpoints/colab_ijepa_cached_epoch15.pt


loading annotations into memory...
Done (t=0.23s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.66s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=20.62s).
Accumulating evaluation results...
DONE (t=4.90s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.194
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.357
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.189
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.057
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.203
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.335
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.200
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.312
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDet

In [ ]:
# --- I-JEPA omurgası (predictive SSL): feature-caching akışı ---
# ⚠️ transformers gerekir; ViT-H ağırlığı HF'ten iner (~2.5 GB). BOYUT CONFOUND'u: ViT-H (632M) vs
# diğerleri ViT-B (~86M) -> sonucu bu notla raporla. Feature-cache ~80 GB olabilir; /content'e
# sığmazsa precompute disk hatası verir (o durumda alttaki alternatife geç ya da küçük subset yap).
!pip install -q transformers

# 1) Trunk feature'larını yaz (ViT-H @518 -> yavaş; ~80 GB disk). Şekil (1280, 37, 37) beklenir:
!python scripts/precompute_features.py --config configs/train_colab_ijepa.yaml --split train

# 2) Neck+head'i cache'den eğit (fp32 -> focal-loss NaN'ından kaçın):
!python scripts/train_cached.py --config configs/train_colab_ijepa.yaml --no-amp

# 3) Değerlendir + 6'lı karşılaştırma:
!python scripts/eval.py --config configs/train_colab_ijepa.yaml --checkpoint checkpoints/colab_ijepa_cached_epoch15.pt
!python scripts/compare_results.py

# --- (Alternatif) cache diske SIĞMAZSA: düz train.py (çok yavaş, ViT-H her adım forward) ---
# !python scripts/train.py --config configs/train_colab_ijepa.yaml --overrides train.amp=false
# !python scripts/eval.py --config configs/train_colab_ijepa.yaml --checkpoint checkpoints/colab_ijepa_epoch15.pt

loading annotations into memory...
Done (t=2.30s)
creating index...
index created!
Loading weights: 100% 517/517 [00:00<00:00, 4264.66it/s]
precompute train: 100% 5625/5625 [48:46<00:00,  1.92it/s]

22500 görsel cache'lendi -> features/colab_ijepa/train
Her biri (1280, 32, 32) float16 ~2.6 MB; toplam ~59.0 GB
Sonraki: python scripts/train_cached.py --config <config>
loading annotations into memory...
Done (t=2.31s)
creating index...
index created!
Loading weights: 100% 517/517 [00:00<00:00, 5471.01it/s]
Config: {'data': {'ann_file': 'data/coco_subset/annotations/instances_train_subset.json', 'img_dir': 'data/coco_subset/images/train', 'val_ann_file': 'data/coco_subset/annotations/instances_val_subset.json', 'val_img_dir': 'data/coco_subset/images/val', 'n_images': 22500, 'img_size': 512, 'num_workers': 4}, 'model': {'backbone_name': 'ijepa_vith14', 'pretrained': True, 'trainable_backbone_layers': 0, 'cls_head_tap': 'fpn_p5'}, 'loss': {'det_cls': 1.0, 'det_box': 1.0, 'seg': 1.0, 'cls': 

## 11. MAE omurgası — ⭐ SWEEP'İN EN ADİL EKLEMESİ (maskeli yeniden-kurma / MIM)
Backbone: **MAE ViT-B/16** (`configs/train_colab_mae.yaml`, `src/mtl/models/mae_backbone.py`). MAE görüntünün %75'ini maskeleyip maskeli patch'lerin **piksellerini yeniden kurmayı** öğrenir — sweep'te temsil edilmeyen **MIM (maskeli yeniden-kurma)** ailesi.

**Neden bu, adil çekirdeğin son parçası:** DINOv1 / CLIP / SAM ile **her eksende birebir aynı** — ViT-B (~86M) · patch16 · img 512 · **grid 32×32** · ImageNet norm (renorm YOK) · SFP neck · aynı head'ler · batch 4 · 16 epoch. **Tek değişken: pretraining sinyali. Sıfır confound.**

> **I-JEPA notu (section 10):** Meta I-JEPA'nın **ViT-B'sini yayınlamadı** (sadece ViT-H/g) → boyut confound'u **düzeltilemez**, ana kıyas tablosuna adil bir satır olarak giremez. "Maskeli tahmin" ailesini **adil temsil eden model MAE'dir**; I-JEPA istenirse "boyut avantajlı bonus" olarak dipnotta kalır.

**Adil çekirdek (hepsi ViT-B/16 @512 → 32×32):** `dino_vitb16` (distillation) · **`mae_vitb16` (maskeli kurma)** · `clip_vitb16` (dil) · `sam_vitb16` (seg-native). ResNet ve DINOv2 bağlam için (bilinen confound'ları RESULTS.md'de notlu).

**Not:** `vit_base_patch16_224.mae` = **saf MAE pretrain** (ImageNet supervised fine-tune YOK) — supervised-ft'li varyant paradigma ayrımını bozardı. timm section 1'de kurulu.

In [ ]:
!git -C /content/mtl pull
%cd /content/mtl


remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 5 (delta 4), reused 5 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 3.13 KiB | 1.04 MiB/s, done.
From https://github.com/itu-itis23-ucgun22/mtl
   fc2694d..596dcf2  dino-backbone -> origin/dino-backbone
Updating fc2694d..596dcf2
Fast-forward
 EXPERIMENTS.md | 26 ++++++++++++++++++++++++++
 RESULTS.md     | 21 +++++++++++++++++++++
 ROADMAP.md     |  2 +-
 3 files changed, 48 insertions(+), 1 deletion(-)
/content/mtl


In [ ]:
import os, glob
print('MTL_WEIGHTS_DIR :', os.environ.get('MTL_WEIGHTS_DIR'))
print('HF_TOKEN set    :', bool(os.environ.get('HF_TOKEN')))
print('\n/content/weights içeriği:')
for f in glob.glob('/content/weights/*'):
    print(f'  {os.path.basename(f)}: {os.path.getsize(f)/1e6:.1f} MB')
print('\nDrive/weights içeriği:')
for f in glob.glob('/content/drive/MyDrive/mtl_data/weights/*'):
    print(f'  {os.path.basename(f)}: {os.path.getsize(f)/1e6:.1f} MB')

In [ ]:
import os, glob, shutil
os.makedirs('/content/weights', exist_ok=True)

# bozuk 0-byte'ları temizle
for f in glob.glob('/content/weights/*.safetensors'):
    if os.path.getsize(f) == 0: os.remove(f)

!cp /content/drive/MyDrive/mtl_data/weights/*.safetensors /content/weights/

for f in sorted(glob.glob('/content/weights/*.safetensors')):
    mb = os.path.getsize(f) / 1e6
    print(f'{"✅" if mb > 100 else "❌ BOZUK"}  {os.path.basename(f)}: {mb:.0f} MB')

os.environ['MTL_WEIGHTS_DIR'] = '/content/weights'
print('\nMTL_WEIGHTS_DIR =', os.environ['MTL_WEIGHTS_DIR'])


In [ ]:
# --- MAE omurgası (maskeli piksel yeniden-kurma / MIM): feature-caching akışı ---
# ⭐ SWEEP'İN EN ADİL EKLEMESİ: ViT-B/16 @512 -> 32x32 grid.
#    DINOv1 / CLIP / SAM ile HER EKSENDE aynı (boyut, patch, grid, norm, neck, head, batch, epoch).
#    TEK DEĞİŞKEN: pretraining. Sıfır confound.
# Not: vit_base_patch16_224.mae = SAF MAE pretrain (ImageNet supervised fine-tune YOK).
#      Renorm gerekmez (MAE ImageNet norm'uyla eğitildi) -> CLIP'teki ek adım burada YOK.

# 1) Trunk feature'larını yaz (ViT-B/16@512 -> şekil (768, 32, 32); ~35 GB):
#!python scripts/precompute_features.py --config configs/train_colab_mae.yaml --split train

# 2) Neck+head'i cache'den eğit (fp32 -> focal-loss NaN'ından kaçın):
#!python scripts/train_cached.py --config configs/train_colab_mae.yaml --no-amp --resume checkpoints/colab_mae_cached_epoch12.pt

# 3) Değerlendir + karşılaştırma:
!python scripts/eval.py --config configs/train_colab_mae.yaml --checkpoint checkpoints/colab_mae_cached_epoch15.pt
!python scripts/compare_results.py

# --- (Alternatif) cache sorun çıkarırsa doğrudan görüntüden eğitim ---
#!python scripts/train.py --config configs/train_colab_mae.yaml --overrides train.amp=false
#!python scripts/eval.py --config configs/train_colab_mae.yaml --checkpoint checkpoints/colab_mae_epoch15.pt

## 11b. DeiT omurgası — supervised ViT (ResNet'in conv/FPN confound'unu kapatır)
Backbone: **DeiT ViT-B/16** (`configs/train_colab_deit.yaml`, `src/mtl/models/deit_backbone.py`). DeiT = ImageNet-1k üzerinde **supervised** (sınıflandırma etiketiyle) eğitilmiş ViT.

**Neden değerli:** Şu ana kadar "supervised" temsilcisi **ResNet**'ti — ama ResNet conv+FPN, diğerleri ViT+SFP → "supervised vs SSL" kıyası **mimari confound'la** karışıktı. DeiT ViT-B/16 olduğu için DINOv1/MAE/CLIP/SAM ile **her eksende aynı** (ViT-B, patch16, 32×32, ImageNet norm, SFP, batch 4) → supervised ayağı da **adilleşir**. **Kilit kıyas: DINOv1 (SSL) vs DeiT (supervised)** = aynı ViT, farklı pretraining hedefi.

**Not:** `deit_base_patch16_224.fb_in1k` = **saf supervised** DeiT (distilled değil — distilled varyant öğretmen-öğrenci kullanır, paradigmayı bulandırırdı). Renorm gerekmez (ImageNet norm).

In [ ]:
# --- DeiT omurgası (supervised ViT): feature-caching akışı ---
# ⭐ Supervised paradigmanın ADİL ViT temsilcisi -> ResNet'in conv/FPN confound'unu kapatır.
#    ViT-B/16 @512 -> 32x32 grid; DINOv1/MAE/CLIP/SAM ile her eksende aynı. Renorm YOK (ImageNet).
# Kilit kıyas: DINOv1 (SSL) vs DeiT (supervised) = aynı ViT, farklı pretraining hedefi.
# Not: deit_base_patch16_224.fb_in1k = SAF supervised (distilled DEĞİL).

# Ağırlık HF'ten inmezse (Colab Xet sorunu): PC'den indir, Drive'a koy, MTL_WEIGHTS_DIR ile yerelden yükle:
#   https://huggingface.co/timm/deit_base_patch16_224.fb_in1k/resolve/main/model.safetensors
#   -> /content/weights/deit_base_patch16_224.fb_in1k.safetensors  (~330 MB)

# 1) Trunk feature'larını yaz ((768, 32, 32); ~35 GB):
!python scripts/precompute_features.py --config configs/train_colab_deit.yaml --split train

# 2) Neck+head'i cache'den eğit (fp32):
!python scripts/train_cached.py --config configs/train_colab_deit.yaml --no-amp



loading annotations into memory...
Done (t=2.71s)
creating index...
index created!
model.safetensors: 100% 346M/346M [00:03<00:00, 103MB/s]
precompute train: 100% 5625/5625 [15:38<00:00,  6.00it/s]

22500 görsel cache'lendi -> features/colab_deit/train
Her biri (768, 32, 32) float16 ~1.6 MB; toplam ~35.4 GB
Sonraki: python scripts/train_cached.py --config <config>
loading annotations into memory...
Done (t=2.70s)
creating index...
index created!
Config: {'data': {'ann_file': 'data/coco_subset/annotations/instances_train_subset.json', 'img_dir': 'data/coco_subset/images/train', 'val_ann_file': 'data/coco_subset/annotations/instances_val_subset.json', 'val_img_dir': 'data/coco_subset/images/val', 'n_images': 22500, 'img_size': 512, 'num_workers': 4}, 'model': {'backbone_name': 'deit_vitb16', 'pretrained': True, 'trainable_backbone_layers': 0, 'cls_head_tap': 'fpn_p5'}, 'loss': {'det_cls': 1.0, 'det_box': 1.0, 'seg': 1.0, 'cls': 0.5}, 'train': {'device': 'cuda', 'batch_size': 4, 'epochs':

In [ ]:
# 3) Değerlendir + karşılaştırma:
!python scripts/eval.py --config configs/train_colab_deit.yaml --checkpoint checkpoints/colab_deit_cached_epoch15.pt
#!python scripts/compare_results.py

loading annotations into memory...
Done (t=0.23s)
creating index...
index created!
Loading and preparing results...
DONE (t=1.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=20.55s).
Accumulating evaluation results...
DONE (t=4.51s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.137
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.278
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.119
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.028
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.133
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.255
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.155
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.250
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDet

MOCO

In [ ]:
!python scripts/precompute_features.py --config configs/train_colab_moco.yaml --split train
!python scripts/train_cached.py --config configs/train_colab_moco.yaml --no-amp
!python scripts/eval.py --config configs/train_colab_moco.yaml --checkpoint checkpoints/colab_moco_cached_epoch15.pt


loading annotations into memory...
Done (t=2.29s)
creating index...
index created!
Traceback (most recent call last):
  File "/content/mtl/scripts/precompute_features.py", line 90, in <module>
    main()
  File "/content/mtl/scripts/precompute_features.py", line 54, in main
    backbone = build_backbone(
               ^^^^^^^^^^^^^^^
  File "/content/mtl/src/mtl/models/backbone.py", line 86, in build_backbone
    return MocoBackbone(pretrained=pretrained, trainable_blocks=trainable_layers)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/mtl/src/mtl/models/moco_backbone.py", line 67, in __init__
    self.vit = _build_moco_vit(pretrained)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/mtl/src/mtl/models/moco_backbone.py", line 49, in _build_moco_vit
    return timm.create_model(MOCO_ARCH, pretrained=True, pretrained_cfg_overlay=dict(hf_hub_id=MOCO_HF), **common)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

BEiT


In [ ]:
!pip install -q -U transformers
!python scripts/precompute_features.py --config configs/train_colab_beit.yaml --split train
!python scripts/train_cached.py --config configs/train_colab_beit.yaml --no-amp
!python scripts/eval.py --config configs/train_colab_beit.yaml --checkpoint checkpoints/colab_beit_cached_epoch15.pt


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 146.6 MB/s eta 0:00:00
loading annotations into memory...
Done (t=2.30s)
creating index...
index created!
HTTP Error 504 thrown while requesting HEAD https://huggingface.co/microsoft/beit-base-patch16-224-pt22k/resolve/main/config.json
Retrying in 1s [Retry 1/5].
HTTP Error 504 thrown while requesting HEAD https://huggingface.co/microsoft/beit-base-patch16-224-pt22k/resolve/main/config.json
Retrying in 2s [Retry 2/5].
HTTP Error 504 thrown while requesting HEAD https://huggingface.co/microsoft/beit-base-patch16-224-pt22k/resolve/main/config.json
Retrying in 4s [Retry 3/5].
HTTP Error 504 thrown while requesting HEAD https://huggingface.co/microsoft/beit-base-patch16-224-pt22k/resolve/main/config.json
Retrying in 8s [Retry 4/5].
HTTP Error 504 thrown while requesting HEAD https://huggingface.co/microsoft/beit-base-patch16-224-pt22k/resolve/main/config.json
Retrying in 8s [Retry 5/5].
HTTP Error 504 thrown while requesting HEAD htt

## 12. Kalitatif inference — model gerçekten çizebiliyor/segmentleyebiliyor mu? (RAPOR için)
Metrikler "ne kadar iyi" der; bu **"göster"** der. `scripts/visualize.py` seçili val görüntülerinde bir veya **birden çok backbone'u yan yana** koşturup her görüntü için tek PNG üretir: **satır 0 = ground truth** (gerçek kutular + maske), sonra **her backbone için** tahmin kutuları | tahmin maskesi, başlıklarda tahmin sınıfları. `--out-dir` Drive olursa kalıcı olur → rapora doğrudan koyabilirsin.

**Not:** model `pretrained=False` ile kurulur (ağırlıklar checkpoint'ten) → backbone **HF'ten İNMEZ**, indirme derdi olmaz. Config'ler aynı val setine baktığı için `--num-images N` tüm backbone'larda **aynı görüntüler** = adil yan-yana kıyas.

In [ ]:
# --- Kalitatif görselleştirme: backbone'ları AYNI görüntülerde yan yana çiz (rapor için) ---
# Çıktı Drive'a -> kalıcı, rapora doğrudan koy. Elindeki checkpoint YOLLARINI kendine göre düzelt.

# En etkili figür: birkaç backbone yan yana (GT + her biri: detection | segmentation)
!python scripts/visualize.py \
    --config     configs/train_colab_dinov2.yaml configs/train_colab_clip.yaml configs/train_colab_sam.yaml configs/train_colab_mae.yaml \
    --checkpoint checkpoints/colab_dinov2_epoch15.pt checkpoints/colab_clip_cached_epoch15.pt checkpoints/colab_sam_cached_epoch15.pt checkpoints/colab_mae_cached_epoch15.pt \
    --num-images 8 --score-thresh 0.3 \
    --out-dir /content/drive/MyDrive/mtl_data/viz/compare

# Tek backbone istersen (daha büyük, tek satır GT + tek satır tahmin):
# !python scripts/visualize.py --config configs/train_colab_dinov2.yaml \
#     --checkpoint checkpoints/colab_dinov2_epoch15.pt --num-images 8 \
#     --out-dir /content/drive/MyDrive/mtl_data/viz/dinov2

# Colab'da birkaç PNG'yi hücre içinde göster:
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('/content/drive/MyDrive/mtl_data/viz/compare/*.png'))[:3]:
    print(f); display(Image(f))

## 13. Epoch-başı metrik eğrileri — öğrenme eğrisi (RAPOR için)
`scripts/plot_metric_curves.py` kaydedilmiş **epoch checkpoint'lerini** tek tek eval'leyip **metrik-vs-epoch** çizer (det_mAP / seg_mIoU / cls_mAP / cls_F1). Bu **loss değil, doğruluk** eğrisi — ve **checkpoint'lerden geriye dönük** çıkarıldığı için runtime resetlense bile (checkpoint'ler Drive'da olduğu sürece) kurtarılır.

**Ne gösterir:** her görevin epoch boyunca nasıl geliştiği, platoya oturup oturmadığı → "16 epoch yetti mi, daha uzun eğitsek artar mıydı?" sorusuna görsel cevap. Model `pretrained=False` (ağırlık checkpoint'ten) → **HF indirmesi yok**. `--max-images` ile hızlanır. Metrikler `<out-dir>/<run>_curve.csv`'ye de yazılır (tekrar eval etmeden yeniden çizersin).

**Ön koşul:** epoch checkpoint'leri (`..._epoch0.pt … _epoch15.pt`) Drive'da durmalı. Sadece son epoch'u tuttuysan eğri tek nokta olur — ara epoch'ları silmediysen tamamı çıkar.

In [ ]:
# --- Epoch-başı metrik eğrisi (öğrenme eğrisi, rapor için) ---
# Her epoch checkpoint'ini eval'ler -> metrik-vs-epoch çizer. glob'u TIRNAK içinde ver.
# Config + checkpoint deseni bir backbone içindir; başka backbone için değiştir.

!python scripts/plot_metric_curves.py \
    --config configs/train_colab_mae.yaml \
    --checkpoints "checkpoints/colab_mae_cached_epoch*.pt" \
    --max-images 500 \
    --out-dir /content/drive/MyDrive/mtl_data/viz/curves

# Çıkan eğriyi göster:
from IPython.display import Image, display
display(Image('/content/drive/MyDrive/mtl_data/viz/curves/colab_mae_curve.png'))

# Diğer backbone'lar için (checkpoint desenini kendine göre değiştir):
# !python scripts/plot_metric_curves.py --config configs/train_colab_dinov2.yaml \
#     --checkpoints "checkpoints/colab_dinov2_epoch*.pt" --max-images 500 \
#     --out-dir /content/drive/MyDrive/mtl_data/viz/curves

## 14. Test seti — ayrı, held-out (Faz 2 final raporlama için)
Şu ana kadar **val** (2000 görüntü) hem doğrulama hem sonuç için kullanıldı. Faz 1 **sabit protokol** olduğu için (val'e göre ayar yapılmadı) mevcut sayılar zaten test kalitesinde. Ama **Faz 2**'de (fine-tune: LoRA rank / kaç katman gibi **seçimler**) düzgün ayrım gerekir: **seçimi val'de yap, final sonucu test'te raporla.**

**Test verisi nereden:** COCO **val2017** (5000 görüntü, tam etiketli). Val subset'i bunun 2000'ini aldı → kalan ~3000'den, **val ile çakışmayan** ~2000'lik bir test subset'i üretiriz (`--exclude` val subset'ini havuzdan çıkarır → val ∩ test = ∅; ikisi de train2017'den zaten ayrı). *COCO `test2017` kullanılamaz — etiketleri public değil, sadece sunucuya submit.*

In [ ]:
# --- Test seti oluştur: val2017'den, val subset'iyle ÇAKIŞMAYAN ~2000 görüntü ---
import os
# Test subset'i üretmek için full val2017 annotations gerekli (yoksa indir)
if not os.path.exists('annotations/instances_val2017.json'):
    !wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip
    !unzip -q -o annotations_trainval2017.zip

# Test JSON: val subset'ini HARİÇ tut -> val ∩ test = boş
!python scripts/prepare_coco_subset.py \
    --ann-file annotations/instances_val2017.json \
    --out data/coco_subset/annotations/instances_test_subset.json \
    --n-images 2000 \
    --exclude data/coco_subset/annotations/instances_val_subset.json

# Test görüntülerini COCO'dan indir (mevcutları atlar). data/coco_subset -> yerel symlink.
!python scripts/download_subset_images.py \
    --ann-file data/coco_subset/annotations/instances_test_subset.json \
    --out-dir data/coco_subset/images/test

# Çakışma kontrolü (0 olmalı):
import json
val = {i['id'] for i in json.load(open('data/coco_subset/annotations/instances_val_subset.json'))['images']}
test = {i['id'] for i in json.load(open('data/coco_subset/annotations/instances_test_subset.json'))['images']}
print(f'val={len(val)}  test={len(test)}  ÇAKIŞMA={len(val & test)}')  # ÇAKIŞMA=0 bekleniyor

In [ ]:
# --- TEST'te değerlendir (val yerine test split; results.csv'ye split='test' etiketiyle) ---
# Faz 2 sonrası final sayı için: val'de seçim yaptıysan test'te raporla.
!python scripts/eval.py --config configs/train_colab_dinov2.yaml \
    --checkpoint checkpoints/colab_dinov2_epoch15.pt \
    --ann-file data/coco_subset/annotations/instances_test_subset.json \
    --img-dir  data/coco_subset/images/test \
    --split-name test

In [ ]:
!python scripts/train.py --config configs/train_colab_mae_lora.yaml --resume checkpoints/colab_mae_lora_step9500.pt
#python scripts/eval.py  --config configs/train_colab_mae_lora.yaml \
       #--checkpoint checkpoints/colab_mae_lora_epoch15.pt


loading annotations into memory...
Done (t=3.17s)
creating index...
index created!
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()

model.safetensors: downloading bytes:  46% 156M/343M [00:01<00:01, 185MB/s, 11.9MB/s  ]
model.safetensors: downloading bytes:  54% 184M/343M [00:01<00:00, 167MB/s, 16.9MB/s  ]
model.safetensors: downloading bytes:  91% 313M/343M [00:02<00:00, 142MB/s, 25.8MB/s  ]
model.safetensors: reconstructing file:  67% 229M/343M [00:02<00:01, 83.9MB/s, 15.1MB/s  ]
model.safetensors: downloading bytes: 100% 326M/326M [00:05<00:00, 57.2MB/s, 27.7M